# Yahoo Finance ingestion - worker

Downloads one series from yfinance, writes it to `finhive.yahoo.<series>`, and exits with a JSON payload the orchestrator appends to `finhive.logs.ingestionLog`.

In [ ]:
dbutils.widgets.text("series", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("job_name", "")

series = dbutils.widgets.get("series")
start_date = dbutils.widgets.get("start_date") or None
job_name = dbutils.widgets.get("job_name")

if not series:
    raise ValueError("widget 'series' is required")

In [ ]:
import json
from datetime import datetime, timezone

import yfinance as yf
from pyspark.sql import functions as F

SOURCE = "Yahoo"

try:
    pdf = yf.download(series, start=start_date, progress=False, auto_adjust=False)
    if pdf.empty:
        raise ValueError(f"yfinance returned no rows for series '{series}'")

    pdf = pdf.reset_index()
    pdf.columns = [
        "_".join(str(p) for p in col if p).lower() if isinstance(col, tuple) else str(col).lower()
        for col in pdf.columns
    ]
    last_observation_date = pdf["date"].max().strftime("%Y-%m-%d")

    ingested_at = datetime.now(timezone.utc)
    sdf = (
        spark.createDataFrame(pdf)
        .withColumn("ingested_at", F.lit(ingested_at))
        .withColumn("pipeline_name", F.lit(job_name))
    )

    spark.sql("CREATE CATALOG IF NOT EXISTS finhive")
    spark.sql("CREATE SCHEMA IF NOT EXISTS finhive.yahoo")

    table_name = f"finhive.yahoo.`{series}`"
    sdf.write.mode("append").saveAsTable(table_name)

    result = {
        "series": series,
        "source": SOURCE,
        "status": True,
        "updateAt": ingested_at.isoformat(),
        "lastObservationDate": last_observation_date,
        "item_count": sdf.count(),
        "error": None,
        "job_name": job_name,
    }
except Exception as e:
    result = {
        "series": series,
        "source": SOURCE,
        "status": False,
        "updateAt": datetime.now(timezone.utc).isoformat(),
        "lastObservationDate": None,
        "item_count": 0,
        "error": str(e),
        "job_name": job_name,
    }

dbutils.notebook.exit(json.dumps(result))